## Comparison with structural isolability measures
* we consider Diagnosability Degree $D^{DEG}$  Structural Isolability Rate (GU, 2023)

In [2]:
import faultdiagnosistoolbox as fdt
import sympy as sym
import numpy as np

### Structural model of four tank system
* while the form of equations (relation between variables and equations) have to be known, there is no need to know the values of the parameters

In [3]:
model_def = {
    'type': 'Symbolic',
    'x': ['F1', 'F2', 'F3', 'F4', 'L1', 'L2', 'L3', 'L4', 'dL1', 'dL2', 'dL3', 'dL4'],  # unknown variables
    'f': ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10'],  # faults
    'z': ['yF1', 'yL1', 'yL4'],  # known variables
    'parameters': ['alpha2', 'alpha3', 'alpha4', 'alpha5', 'S2', 'S3', 'S4', 'S5', 'g']  # parameters
}

# Create symbolic variables for the unknowns, knowns, and parameters
sym.var(model_def['x'])
sym.var(model_def['f'])
sym.var(model_def['z'])
sym.var(model_def['parameters'])

# Define the relationships using the equations provided
model_def['rels'] = [
    fdt.DiffConstraint('dL1', 'L1'),
    fdt.DiffConstraint('dL2', 'L2'),
    fdt.DiffConstraint('dL3', 'L3'),
    fdt.DiffConstraint('dL4', 'L4'),
    
    -dL1 + F1 - F2 + f1,
    -dL2 + F2 - F3 + f2,
    -dL3 + F3 - F4 + f3,
    -dL4 + F4 - alpha5 * S5 * sym.sqrt(2 * g * L4) + f4,
    
    -F2 + alpha2 * S2 * sym.sqrt(2 * g * (L1 - L2)) + f5,
    -F3 + alpha3 * S3 * sym.sqrt(2 * g * (L2 - L3)) + f6,
    -F4 + alpha4 * S4 * sym.sqrt(2 * g * (L3 - L4)) + f7,
    
    # Sensor equations
    -F1 + yF1 + f8,
    -L1 + yL1 + f9,
    -L4 + yL4 + f10,
]

In [4]:
model = fdt.DiagnosisModel(model_def)
model.Lint()

Model information

  Type:Symbolic, dynamic

  Variables and equations
    12 unknown variables
    3 known variables
    10 fault variables
    14 equations, including 4 differential constraints

  Degree of redundancy: 2
  Degree of redundancy of MTES set: 1


  Model validation finished with 0 errors and 0 warnings.


### The set of Minimally Structurally Overdetermined Sets (MSOs)

In [5]:
msos = model.MSO()
msos

[[13, 12, 10, 9, 8, 3, 7, 2, 6, 1, 5],
 [13, 12, 10, 9, 8, 3, 7, 2, 6, 0, 4, 11],
 [13, 12, 10, 9, 8, 3, 7, 1, 5, 0, 4, 11],
 [13, 12, 10, 9, 8, 2, 6, 1, 5, 0, 4, 11],
 [13, 12, 10, 9, 3, 7, 2, 6, 1, 5, 0, 4, 11],
 [13, 12, 10, 8, 3, 7, 2, 6, 1, 5, 0, 4, 11],
 [13, 12, 9, 8, 3, 7, 2, 6, 1, 5, 0, 4, 11],
 [13, 10, 9, 8, 3, 7, 2, 6, 1, 5, 0, 4, 11],
 [12, 10, 9, 8, 3, 7, 2, 6, 1, 5, 0, 4, 11]]

There is a following correspondence between the residuals used in the paper and the MSOs:

* $r_1$ - msos[8]
* $r_2$ - msos[7]
* $r_3$ - msos[0]

The correspondence is based on known variables. Please note, structural isolability indices are based on the whole set of residuals.

### Fault Signature Matrices

#### FSM based on the full set of MSOs

In [6]:
fsm = model.FSM(msos)
fsm

array([[0, 1, 1, 1, 1, 1, 1, 0, 1, 1],
       [1, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 0, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 0, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]])

#### FSM based only on the MSOs corresponding to the residuals used in the paper
* it is the same as FSM based on expert knowledge presented in Table 10 in the paper, showing the close correspondence of the approaches

In [7]:
msos_paper = [msos[i] for i in [8, 7, 0]]
fsm_paper = model.FSM(msos_paper)
fsm_paper

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
       [1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
       [0, 1, 1, 1, 1, 1, 1, 0, 1, 1]])

### All faults are structurally detectable

In [9]:
model.DetectabilityAnalysis()

(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10'], [])

### Structural isolability analysis
* 1 denotes faults that are *not* isolable
* therefore only the pair of faults $\{f_1, f_8\}$ is not structurally isolable
* therefore isolabiility degree equals $D^{DEG} = \frac{D_c}{card(F)} = \frac{9}{10}$ 

In [12]:
model.IsolabilityAnalysis()

array([[1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]])

### Structural Isolability Rate (Gu, 2023)
* is defined as a number of faults that can be isolated form each other faults
* which is equal to the number of rows in Isolability Matrix with sum equal 1

In [9]:
im = model.IsolabilityAnalysisFSM(fsm)
im

array([[1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]])

In [10]:
def structural_isolability_ratio(im):
    # equal failure rates assumed
    count = np.sum(np.sum(im, axis=1) <= 1)
    return count / im.shape[0]

Assuming equal failure rates we obtain $SIR = 0.8$

In [11]:
structural_isolability_ratio(im)

np.float64(0.8)